In [1]:
import os
import platform
import sys
import time

from dataclasses import asdict

import numpy as np
import h5py
import pint
import tables

from pint import Quantity

In [2]:
match platform.system():
    case "Linux":
        sys.path.insert(1, os.path.abspath(".."))
        import lysis
        from lysis.util import Q_
        lysis_root = os.path.join("/", "home", "bpaynter", "git", "UCO-OpResearch", "lysis")
    case "Windows":
        import src.python.lysis as lysis

In [3]:
#Need to make considerations for loading the data from an existing HDF5 instead of just loading a new one every time.
#Need to start moving this notebook to lysis\src\python\lysis\util\ and replace datastore.py with this code becoming methods.
#Need to create some sort of class
run_code="2024-09-02-1411"
r = lysis.util.Run(os.path.join(lysis_root, "data"), run_code=run_code)
r.read_file()
r.macro_params.total_molecules

/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/parameters.py:279: RuntimeWarning: Run parameter file does not contain Microscale parameters. Using defaults.
  warnings.warn(
/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/parameters.py:307: RuntimeWarning: Parameter pore_size has no units. Assuming centimeters.
  warnings.warn(
/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/parameters.py:307: RuntimeWarning: Parameter diffusion_coeff has no units. Assuming cm^2/s.
  warnings.warn(
/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/parameters.py:307: RuntimeWarning: Parameter total_time has no units. Assuming seconds.
  warnings.warn(
/home/bpaynter/git/UCO-OpResearch/lysis/src/python/lysis/util/parameters.py:307: RuntimeWarning: Parameter save_interval has no units. Assuming sec.
  warnings.warn(


21105

In [4]:
data = lysis.util.DataStore(run_code, os.path.join(lysis_root, "data"), 'r')

In [5]:
data.export_fortran_micro_data(os.path.join(lysis_root, "data", run_code), "__hdf5")

In [ ]:
# r.to_file()

Creates a variable f that references the file we want to access.
Need to change variables according to the name of you .h5 file

In [ ]:
h5py.get_config().track_order = True
h5file = h5py.File(os.path.join(lysis_root, "data", f'{run_code}.h5'), 'w')

In [ ]:
tpa_molecule_status = h5py.enum_dtype({i.name: i.value for i in lysis.util.CONST.MOL_STATUS}, basetype="u1")

In [ ]:
tpa_bind_event_type = np.dtype([
    ("Simulation Time Elapsed", np.float64), 
    ("tPA Molecule Index", np.int64), 
    ("Molecule New Status", tpa_molecule_status),
    ("Grid Location Index", np.int16)
    ])

fiber_degrade_event_type = np.dtype([
    ("Simulation Time Elapsed", np.float64),
    ("Grid Location Index", np.int32),
    ("Fiber New Degrade Time", np.float64),
])

Creates folder and data set structure

In [ ]:
# runs = h5file.create_group("1-PKd") 
micro_data = h5file.create_group("micro_data") 
macro_data = h5file.create_group("macro_data")



#Micro data set initializations
pli_first_time = micro_data.create_dataset("pli_first_time", (r.micro_params.simulations, ), dtype= np.float64, compression="gzip")
tpa_final_num = micro_data.create_dataset("tpa_final_num", (r.micro_params.simulations, ), dtype= np.uint8, compression="gzip")
fiber_degraded = micro_data.create_dataset("fiber_degraded", (r.micro_params.simulations, ), dtype= np.bool_, compression="gzip")
sim_final_time = micro_data.create_dataset("sim_final_time", (r.micro_params.simulations, ), dtype= np.float64, compression="gzip")
pli_generated_num = micro_data.create_dataset("pli_generated_num", (r.micro_params.simulations, ), dtype= np.uint16, compression="gzip")
tpa_leaving_time = micro_data.create_dataset("tpa_leaving_time", (r.micro_params.simulations, ), dtype= np.float64, compression="gzip")
tpa_unbound_by_pli = micro_data.create_dataset("tpa_unbound_by_pli", (r.micro_params.simulations, ), dtype= np.bool_, compression="gzip")
tpa_unbound_kinetic = micro_data.create_dataset("tpa_unbound_kinetic", (r.micro_params.simulations, ), dtype = np.bool_, compression="gzip")

#Macro data set initializations


for i in range(0 , r.macro_params.simulations):
    simulation = macro_data.create_group(f"sim_{i:02}")
    fiber_degrade_time = simulation.create_dataset("fiber_degrade_time", (1,) , dtype=fiber_degrade_event_type, maxshape = (None,), compression="gzip", chunks=(10_000,))
    tpa_bind_events = simulation.create_dataset("tpa_bind_events", (1,) , dtype=tpa_bind_event_type , maxshape = (None,), compression="gzip", chunks=(10_000,)) #snapshot time = 
    snapshot_time = simulation.create_dataset("snapshot_time", (1,) , dtype=np.float64 , maxshape = (None,), compression="gzip", chunks=(10_000,))
    #tpa_location_snapshot = simulation.create_dataset("tpa_location_snapshot", (r.macro_params.total_molecules , 1) , dtype=np.int32 , maxshape = (r.macro_params.total_molecules, None)) #I uncapped the max number of rows so the data can fit
    tpa_location_snapshot = simulation.create_dataset("tpa_location_snapshot", (r.macro_params.total_molecules, 1) , dtype=np.int32 , maxshape = (r.macro_params.total_molecules, None), compression="gzip", chunks=(r.macro_params.total_molecules, 5))
    tpa_transit_time = simulation.create_dataset("tpa_transit_time", (r.macro_params.total_molecules,) , dtype=np.float64, compression="gzip")
    


Reads in Micro scale data into file system

In [ ]:
micro_file_code = "_PLG2_tPA01_TB-xiii"

pli_first_time[:] = np.fromfile(
    os.path.join(r.os_path, f"firstPLi{micro_file_code}.dat"),
)
pli_first_time.attrs["units"] = "seconds"

tpa_final_num[:] = np.fromfile(os.path.join(r.os_path, f"lasttPA{micro_file_code}.dat"), dtype=np.int32)
tpa_final_num.attrs["units"] = "none"

fiber_degraded[:] = np.fromfile(
    os.path.join(r.os_path, f"lyscomplete{micro_file_code}.dat"), 
    dtype=np.int32
).astype(bool)
fiber_degraded.attrs["units"] = "none"

sim_final_time[:] = np.fromfile(os.path.join(r.os_path, f"lysis{micro_file_code}.dat"))
sim_final_time.attrs["units"] = "seconds"

pli_generated_num[:] = np.fromfile(os.path.join(r.os_path, f"PLi{micro_file_code}.dat"), dtype=np.int32).astype(np.uint16)
pli_generated_num.attrs["units"] = "none"

tpa_leaving_time[:] = np.fromfile(os.path.join(r.os_path, f"tPA_time{micro_file_code}.dat"))
tpa_leaving_time.attrs["units"] = "seconds"

tpa_unbound_by_pli[:] = np.fromfile(
    os.path.join(r.os_path, f"tPAPLiunbd{micro_file_code}.dat"), 
    dtype=np.int32
).astype(bool)
tpa_unbound_by_pli.attrs["units"] = "none"

tpa_unbound_kinetic[:] = np.fromfile(
    os.path.join(r.os_path, f"tPAPLiunbd{micro_file_code}.dat"), 
    dtype=np.int32
).astype(bool)
tpa_unbound_kinetic.attrs["units"] = "none"

Reads in Macro scale Fiber Degrade Time Data

In [ ]:
macro_file_code = f"_TB-xiii__21_105"
for i in range (0 , 10):
    file_reference = h5file[f"macro_data/sim_{i:02}/fiber_degrade_time"]
    data = np.loadtxt(os.path.join(r.os_path, f"{i:02}", f"f_deg_list{macro_file_code}_{i:02}.dat") , delimiter=",", dtype=fiber_degrade_event_type)
    # The data in column 1 is 1 indexed, so we need to convert it to 0 indexed
    data[:]["Grid Location Index"] = data[:]["Grid Location Index"] - 1   
    file_reference.resize(data.shape)
    file_reference[:] = data
    #Adding Attributes to datasets
    file_reference.attrs["units"] = ["seconds" , "none" , "seconds"]

Reads in Macro scale TPA Bind Events Data | ~~Needs to be reworked for efficiency 51 seconds (5.5 min on Buddy)~~ Fixing chunk size took care of this

I believe what is taking so long is the resizing of the HDF dataset that is going on. Perhaps by specifying size on instatiation, we can avoid this.

It was the writing to the HDF itself. When resizing is enabled, chunking is enabled. With the initial size set to '1', the chunk size was also set to 1. This made writing horribly inefficient as it created a new chunk for each row.

In [ ]:
for i in range (0 , 10):
    file_reference = h5file[f"macro_data/sim_{i:02}/tpa_bind_events"]
    data = np.loadtxt(os.path.join(r.os_path, f"{i:02}", f"m_bind_t{macro_file_code}_{i:02}.dat") , delimiter=",", dtype=tpa_bind_event_type)
    # The data in column 1 and 3 is 1 indexed, so we need to convert it to 0 indexed
    data[:]['tPA Molecule Index'] = data[:]['tPA Molecule Index'] - 1
    data[:]["Grid Location Index"] = data[:]["Grid Location Index"] - 1
    file_reference.resize(data.shape)
    file_reference[:] = data
    file_reference.attrs["units"] = ["seconds" , "none" , "none", "none"]
    

Reads in Macro scale snapshot time data

In [ ]:
for i in range (0 , 10):
    file_reference = h5file[f"macro_data/sim_{i:02}/tpa_location_snapshot"]
    data = np.fromfile(os.path.join(r.os_path, f"{i:02}", f"m_loc{macro_file_code}_{i:02}.dat") , dtype = np.int32).reshape(r.macro_params.total_molecules, -1)
    data = data - 1 #The data is 1 indexed, so we need to convert it to 0 indexed
    file_reference.resize(data.shape)
    file_reference[:] = data
    file_reference.attrs["units"] = 'none'

Reads in Macro scale TPA Transit Time Data

In [ ]:
for i in range (0 , 10):
    file_reference = h5file[f"macro_data/sim_{i:02}/tpa_transit_time"]
    data = np.fromfile(os.path.join(r.os_path, f"{i:02}", f"mfpt{macro_file_code}_{i:02}.dat") , dtype = np.float64) #.reshape(-1,1)
    file_reference[:] = data
    file_reference.attrs["units"] = "seconds"

Reads in Macro scale Snapshot Time Data

In [ ]:
for i in range (0 , 10):
    file_reference = h5file[f"macro_data/sim_{i:02}/snapshot_time"]
    data = np.fromfile(os.path.join(r.os_path, f"{i:02}", f"tsave{macro_file_code}_{i:02}.dat") , dtype = np.float64) #.reshape(-1,1)
    file_reference.resize(data.shape)
    file_reference[:] = data
    file_reference.attrs["units"] = "seconds"

Adding units to group attributes

In [ ]:
micro_group = h5file["micro_data"]
micro_group.attrs["data_version"] = "2.0.0"
units = lysis.util.MicroParameters.units()
for k, v in asdict(r.micro_params).items():
    if isinstance(v, Quantity):
        micro_group.attrs[k] = str(v.to(units[k]))
    else:
        micro_group.attrs[k] = v



In [ ]:
macro_group = h5file["macro_data"]
macro_group.attrs["data_version"] = "2.0.0"
units = lysis.util.MacroParameters.units()
for k, v in asdict(r.macro_params).items():
    if isinstance(v, Quantity):
        macro_group.attrs[k] = str(v.to(units[k]))
    else:
        macro_group.attrs[k] = v

Create group for log files

In [ ]:
log_group = h5file.create_group("log_files")
with open(os.path.join(r.os_path, f"micro{micro_file_code}.txt"), 'r') as file:
    micro_log = file.readlines()
micro_log_dataset = log_group.create_dataset("micro_log", (1,) , dtype=h5py.string_dtype() , maxshape = (None,), compression="gzip", chunks=(10_000,))
micro_log_dataset.resize((len(micro_log),))
micro_log_dataset[:] = micro_log

In [ ]:
for i in range(r.macro_params.simulations):
    macro_log_dataset = log_group.create_dataset(f"macro_log__sim_{i:02}", (1,) , dtype=h5py.string_dtype() , maxshape = (None,), compression="gzip", chunks=(10_000,))
    with open(os.path.join(r.os_path, f"{i:02}", f"macro{macro_file_code}_{i:02}.txt"), 'r') as file:
        macro_log = file.readlines()
    macro_log_dataset.resize((len(macro_log),))
    macro_log_dataset[:] = macro_log

In [ ]:
h5file.close()

Test code for "Micro to Macro" conversions.

In [ ]:
h5file = h5py.File(os.path.join(lysis_root, "data", f'{run_code}.h5'), 'r')

In [ ]:
micro_data = h5file["micro_data"]
set_size = micro_data["pli_first_time"].size // 100
tPAleave = np.append(np.arange(0, 1, 0.01), [1.0])
np.savetxt(os.path.join(r.os_path, "tPAleave_numpy.dat"), tPAleave)

In [ ]:
# tPA_leave_time = np.fromfile(os.path.join(e.os_path, f"tPA_time_{file_code}.dat"))
indices = micro_data["tpa_leaving_time"][:].argsort()
tsectPA = np.append(
    [0], micro_data["tpa_leaving_time"][:][indices[set_size - 1 :: set_size]]
)
np.savetxt(os.path.join(r.os_path, "tsectPA_numpy.dat"), tsectPA)

In [ ]:
lysis_complete = micro_data["fiber_degraded"][:]
lysis_time = micro_data["sim_final_time"][:]
lysis_time[~lysis_complete] = 6000
lysismat = np.stack(
    [
        np.sort(lysis_time[indices[i * set_size : (i + 1) * set_size]])
        for i in range(100)
    ]
).T
np.savetxt(os.path.join(r.os_path, "lysismat_numpy.dat"), lysismat)

In [ ]:
lenlysisvect = lysismat.argmax(axis=1)+1
np.savetxt(os.path.join(r.os_path, "lenlysisvect_numpy.dat"), lenlysisvect)

In [ ]:
h5file.close()

Post Processing

In [ ]:
import pint
u = pint.UnitRegistry()
Q = u.Quantity

In [4]:
h5file = h5py.File(os.path.join(lysis_root, "data", f'{run_code}.h5'), 'r')

In [ ]:
for k, v in h5file.items():
    print(k)

In [ ]:
H5_dataset = h5file["macro_data/sim_00/fiber_degrade_time"]
unit_array = H5_dataset.attrs["units"]
dataset = np.array(h5file["macro_data/sim_00/fiber_degrade_time"])
event_time = dataset[:,0]
legs2 = [400.0, 300.0] * u.centimeter
legs2 = event_time * u(unit_array[0])
print(legs2.to('min'))
#print(legs2)
unit_array

In [ ]:
data = h5file["micro_data/tpa_final_num"]
data.attrs["units"]

In [ ]:
r.macro_params

In [ ]:
h5py.version.version

In [ ]:
with open("test.txt", 'r') as file:
    file.write("Test2")